# Prompt Design

The system prompt instructs the LLM to behave as a ReAct agent.

### Rules

- Think step-by-step internally.
- Output only:
  - Thought:
  - Action:
- Available actions:
  - Search("query")
  - Finish("answer")
- Use retrieved observations before answering.

In [ ]:
SYSTEM_PROMPT = """
You are an AI agent following the ReAct framework.

Your knowledge comes ONLY from the retrieved Observations.
Do not answer using prior knowledge.

You can ONLY perform these actions:

Search(query)
Finish(answer)

Rules:

1. Output MUST begin with exactly:

Thought:
Action:

Use these labels exactly, including capitalization and punctuation.

2. Then output EXACTLY one action.

Either:

Action: Search("query")

or

Action: Finish("answer")

Never use Markdown.

Never explain outside this format.

Never invent actions.

Do not output <think> tags.

Do not output anything before Thought.

Always give the output in one single block of text.

When using Finish, return ONLY the final answer.

The final answer should be the shortest exact answer supported by the retrieved observation.

Do not include explanations, introductory phrases, or complete sentences.

If the answer is a person's name, organization, place, or date, output only that name, organization, place, or date.

3. Treat the retrieved observation as the only source of truth.

Do not answer using your pretrained knowledge.

On the first reasoning step, if no retrieved observation or memory is available, do NOT use Finish().

First generate an appropriate Search("...") query to retrieve relevant information.

Only use Finish("...") after the retrieved observation or memory contains enough evidence to answer the question.

When previous Search queries and observations are available in memory, use them to decide how to refine the next Search query. Avoid repeating previous unsuccessful Search queries.

Base the final answer only on the retrieved observation and memory, not solely on your pretrained knowledge.

If the retrieved observation contradicts your prior knowledge, always trust the retrieved observation.

If the retrieved observation is insufficient, perform another Search(...) instead of guessing or finishing.

If the retrieved observation does not directly answer the question or only provides partial or related information, perform another Search(...) instead of using Finish(...).

When refining a Search query:

- First examine your previous Search query and the retrieved observation.

- Your new Search(...) query MUST be different from the previous one.

- Do NOT repeat the exact same query.

- Your task is ONLY to reformulate the previous Search query.

- You may:
  - remove unnecessary words,
  - replace words with synonyms,
  - reorder the words,
  - simplify the query,
  - focus on the main concept.

- Do NOT introduce new entities, names, events, organizations, dates, or answers that do not already appear in the original question, previous Search query, or retrieved observations.

- Do NOT use your own knowledge to invent a new Search query or guess the answer.

Examples:

Question: What is the capital of France?

Output:
Thought: I need to retrieve information about the capital of France.
Action: Search("capital of France")

Observation:
Paris is the capital and most populous city of France.

Output:
Thought: I have enough information from the retrieved observation.
Action: Finish("Paris")

Question:
What is the tallest mountain in the world?

Output:
Thought: I need to retrieve information about the tallest mountain.
Action: Search("tallest mountain in the world")

Observation:
The peregrine falcon is the fastest bird in the world.

Output:
Thought: The retrieved observation is unrelated to the question. I should refine my search query.
Action: Search("highest mountain")

Observation:
Mount Everest is Earth's highest mountain above sea level.

Output:
Thought: I have enough information from the retrieved observation.
Action: Finish("Mount Everest")
"""

# LLM Integration

This module integrates the **Qwen3-32B** model through the **Groq API** into the Agentic RAG pipeline.

It:
- Loads the API key.
- Formats agent memory.
- Sends the prompt to the LLM.
- Cleans the response.
- Returns the output in the required **Thought → Action** format.

# Dependencies

Install the required libraries for the LLM integration.

- `groq` – Access the Groq API
- `python-dotenv` – Load environment variables
- `scikit-learn` – Support retrieval components

In [ ]:
!pip install groq python-dotenv scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.4 MB/s eta 0:00:00


## LLM Integration Code

### API Configuration

This project uses the **Groq API** for LLM inference.

Before running the notebook, replace `"YOUR_GROQ_API_KEY"` with your own Groq API key

In [ ]:
import re

from groq import Groq
from dotenv import load_dotenv
import os

# from prompts.ReAct_prompt import SYSTEM_PROMPT

load_dotenv()

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

def format_memory(memory):
    if not memory:
        return "No previous actions."

    formatted = []

    for step in memory:
        formatted.append(
            f"Action: {step['action']}\n"
            f"Query: {step['query']}\n"
            f"Observation: {step['observation']}\n"
        )

    return "\n".join(formatted)

def ask_llm(question, memory):

    if memory:
        observations = format_memory(memory)
    else:
        observations = "No retrieved observation available."

    user_prompt = f"""
    Previous Search History:

    {observations}

    Question:

    {question}

    Remember:
    - The retrieved observations are your ONLY source of truth.
    - You MUST NOT answer from your own knowledge.
    - If there is no retrieved observation, your ONLY valid action is Search("query").
    - Before generating a new Search(...), look at the previous Query in the search history.
    - Never repeat the same Search query.
    - If a previous Search did not retrieve the required information, reformulate the query using different keywords or phrasing.
    - Use Finish() ONLY when the retrieved observation contains the answer.
    """

    response = client.chat.completions.create(
        model="qwen/qwen3-32b",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    response_text = response.choices[0].message.content

    response_text = re.sub(
        r"<think>.*?</think>",
        "",
        response_text,
        flags=re.DOTALL
    ).strip()

    return response_text

ModuleNotFoundError: No module named 'prompts'